# 🛒 ShopPulse — Notebook 01: Data Preprocessing

**Goal**: Clean the Online Retail dataset for analysis.

**Steps**:
1. Load raw data (real or synthetic)
2. Handle missing CustomerIDs
3. Remove cancelled orders (InvoiceNo starting with 'C')
4. Remove negative quantities and zero prices
5. Drop duplicate rows
6. Parse dates and create derived features
7. Exploratory overview of clean data

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
PURPLE = '#7C3AED'
CYAN   = '#06B6D4'

print('Libraries loaded ✅')

In [ ]:
from data_loader import load_raw_data

raw = load_raw_data()
print(f'Raw data shape: {raw.shape}')
raw.head()

In [ ]:
# ── 1. Missing values ──────────────────────────────────────────────────────────
print('=== Missing Values ===')
missing = raw.isnull().sum()
missing_pct = (missing / len(raw) * 100).round(2)
pd.DataFrame({'Missing': missing, 'Pct': missing_pct}).query('Missing > 0')

In [ ]:
# ── 2. Cancellations ──────────────────────────────────────────────────────────
cancelled = raw[raw['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Cancelled orders: {len(cancelled):,} ({len(cancelled)/len(raw)*100:.1f}%)')
print(f'Normal orders:    {len(raw) - len(cancelled):,}')

In [ ]:
# ── 3. Negative quantities / zero prices ────────────────────────────────────
raw_no_cancel = raw[~raw['InvoiceNo'].astype(str).str.startswith('C')]
neg_qty   = raw_no_cancel[raw_no_cancel['Quantity'] <= 0]
zero_price = raw_no_cancel[raw_no_cancel['UnitPrice'] <= 0]
print(f'Negative/zero quantity rows: {len(neg_qty):,}')
print(f'Zero price rows:             {len(zero_price):,}')

In [ ]:
# ── 4. Full clean pipeline ─────────────────────────────────────────────────
from data_loader import clean_data, get_clean_data

print('Running full cleaning pipeline...')
df = get_clean_data()
print(f'\nFinal dataset: {df.shape}')
df.info()

In [ ]:
# ── 5. Describe cleaned data ─────────────────────────────────────────────────
df[['Quantity', 'UnitPrice', 'Revenue']].describe().round(3)

In [ ]:
# ── 6. Time range ──────────────────────────────────────────────────────────
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'Unique customers:  {df["CustomerID"].nunique():,}')
print(f'Unique products:   {df["StockCode"].nunique():,}')
print(f'Unique countries:  {df["Country"].nunique():,}')
print(f'Total revenue:     £{df["Revenue"].sum():,.2f}')

In [ ]:
# ── 7. Monthly order counts ─────────────────────────────────────────────────
monthly = df.groupby('YearMonth')['InvoiceNo'].nunique()

fig, ax = plt.subplots(figsize=(12, 4), facecolor='#0F0F1A')
ax.set_facecolor('#0F0F1A')
monthly.plot(kind='bar', ax=ax, color=PURPLE, alpha=0.85)
ax.set_title('Monthly Order Count (after cleaning)', color='white', fontsize=14, pad=12)
ax.set_xlabel('Month', color='white')
ax.set_ylabel('Orders', color='white')
ax.tick_params(colors='white')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()